# Fase 7 - Reels cenital v1, v2 (Instagram, 30s)

Genera **2 reels** cenital para elegir (`reel_cenital_v1`, `reel_cenital_v2`), ambos **30s vertical 1080x1920**.

- Broadcast con **bounding boxes (sin nombre)** en robots + balon (`draw_boxes=True, box_labels=False`)
- Audio: **crowd** del MOV original + **silbato** de inicio (primeros segundos)
- Tolerante: si falta fuente o audio, salta/queda mudo (no rompe)

Card titulo (2.5s) + contenido (25s) + card cierre (2.5s) = 30s exactos.

In [ ]:
import subprocess
from pathlib import Path

ROOT = Path('/workspace/FutBotMX-UAQTeam')
OUT  = ROOT / 'notebooks/fase_7_visuales/outputs'
TMP  = OUT / '_reel_tmp'
ADIR = ROOT / 'notebooks/fase_7_visuales/audio'
TMP.mkdir(parents=True, exist_ok=True)
FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
WHISTLE = ADIR / 'whistle.wav'

W, H = 1080, 1920
CARD, CONTENT = 2.5, 25.0      # 2.5 + 25 + 2.5 = 30s
START = 10.0

STEMS = [
    ('IMG_9933_5m30', 'v1', 10.0),
    ('IMG_9938_min1', 'v2', 10.0),
]
print('whistle:', WHISTLE.exists(), '| audios:', len(list(ADIR.glob('audio_*.m4a'))))

In [ ]:
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2500:]); raise RuntimeError('ffmpeg fail')

def probe(path, key):
    return subprocess.run(['ffprobe','-v','error','-select_streams','v:0',
        '-show_entries',f'stream={key}','-of','csv=p=0',str(path)],
        capture_output=True,text=True).stdout.strip()

def dtxt(text, y, size, color='white', alpha=0.6):
    t = text.replace('\\','').replace(':','\\:').replace("'",'\u2019')
    return (f"drawtext=fontfile={FONT}:text='{t}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:box=1:boxcolor=black@{alpha}:boxborderw=16")

def make_card(path, lines):
    vf = f'color=c=0x0a0a0a:s={W}x{H}:d={CARD}:r=30'
    for (txt,y,sz,col) in lines:
        vf += ',' + dtxt(txt,y,sz,col)
    run(['ffmpeg','-y','-f','lavfi','-i',vf,'-frames:v',str(int(CARD*30)),
         '-r','30','-c:v','libx264','-pix_fmt','yuv420p',str(path)])

In [ ]:
# Cards comunes
card_t = TMP / 'cen_card_title.mp4'
make_card(card_t, [
    ('FutBotMX 2026', H//2 - 200, 96, 'white'),
    ('UAQ Team  -  Broadcast', H//2 - 50, 60, '0x39d98a'),
    ('Segmentacion - Eventos - Kalman - Minimapa', H//2 + 90, 40, '0xcccccc'),
])
card_o = TMP / 'cen_card_out.mp4'
make_card(card_o, [
    ('SAM 3 + YOLO + ByteTrack + Kalman', H//2 - 110, 50, 'white'),
    ('Marcador - Posesion - Heatmap - Velocidad', H//2, 44, '0x39d98a'),
    ('github.com/RodMed0709/FutBotMX-UAQTeam', H//2 + 150, 36, '0xaaaaaa'),
])
print('cards OK')

In [ ]:
def build_content(src, dst, start):
    SW, SH = int(probe(src,'width')), int(probe(src,'height'))
    fg_h = int(round(W*SH/SW/2)*2)
    bar = (H - fg_h)//2
    top1 = dtxt('FutBotMX 2026  -  UAQ Team', max(40, bar//2-70), 48)
    top2 = dtxt('Broadcast en vivo', max(120, bar//2+10), 38, '0x39d98a')
    by = H - bar + max(40, bar//2-80)
    bot1 = dtxt('Boxes robots + balon  -  Segmentacion + Eventos', by, 36, '0x39d98a')
    bot2 = dtxt('Minimapa - Heatmap - Velocidad (Kalman) - Marcador', by+66, 30, '0xdddddd')
    fc = (f'[0:v]split=2[b][f];'
          f'[b]scale=-2:{H},crop={W}:{H},boxblur=22:2,eq=brightness=-0.22:saturation=0.5[bg];'
          f'[f]scale={W}:{fg_h},eq=contrast=1.05:saturation=1.12[fg];'
          f'[bg][fg]overlay=(W-w)/2:(H-h)/2[v0];'
          f'[v0]{top1}[v1];[v1]{top2}[v2];[v2]{bot1}[v3];[v3]{bot2}[outv]')
    run(['ffmpeg','-y','-ss',str(start),'-t',str(CONTENT),'-i',str(src),
         '-filter_complex',fc,'-map','[outv]','-r','30',
         '-c:v','libx264','-pix_fmt','yuv420p',str(dst)])

def mux_audio(video_in, final, crowd, start):
    dly = int(CARD*1000)
    have_crowd = crowd.exists(); have_wh = WHISTLE.exists()
    if not have_crowd and not have_wh:
        import shutil; shutil.copy(video_in, final); return 'mudo'
    inputs = ['-i', str(video_in)]; parts, mixn = [], []; idx = 1
    if have_crowd:
        inputs += ['-i', str(crowd)]
        parts.append(f'[{idx}:a]atrim={start}:{start+CONTENT},asetpts=PTS-STARTPTS,volume=1.5,adelay={dly}|{dly}[cr]')
        mixn.append('[cr]'); idx += 1
    if have_wh:
        inputs += ['-i', str(WHISTLE)]
        parts.append(f'[{idx}:a]volume=1.1,adelay=500|500[wh]')
        mixn.append('[wh]'); idx += 1
    fc = ';'.join(parts) + ';' + ''.join(mixn) + f'amix=inputs={len(mixn)}:duration=longest:normalize=0,apad,atrim=0:30,afade=t=out:st=29:d=1[aout]'
    run(['ffmpeg','-y',*inputs,'-filter_complex',fc,'-map','0:v','-map','[aout]',
         '-c:v','copy','-c:a','aac','-b:a','160k',str(final)])
    return 'crowd+silbato' if have_crowd else 'silbato'

In [ ]:
results = []
for stem, vlab, st in STEMS:
    src = OUT / f'broadcast_box_{stem}.mp4'
    if not src.exists():
        print('SKIP (falta fuente):', src.name); continue
    content = TMP / f'content_{vlab}.mp4'
    build_content(src, content, st)
    silent = TMP / f'silent_{vlab}.mp4'
    lst = TMP / f'concat_{vlab}.txt'
    lst.write_text('\n'.join(f"file '{p}'" for p in [card_t, content, card_o]) + '\n')
    run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(lst),
         '-r','30','-c:v','libx264','-pix_fmt','yuv420p',str(silent)])
    crowd = ADIR / f'audio_{stem}.m4a'
    final = OUT / f'reel_cenital_{vlab}_{stem}.mp4'
    snd = mux_audio(silent, final, crowd, st)
    dur = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',str(final)],capture_output=True,text=True).stdout.strip()
    results.append((vlab, final.name, dur, snd))
    print(f'{vlab}: {final.name}  dur={dur}s  audio={snd}')

print('\n===== REELS CENITAL =====')
for vlab, name, dur, snd in results:
    print(f'  {vlab}  {name}  ({dur}s, {snd})')